# Descriptives: simulated demand datasets

Three simulated datasets in `code/build/input/public/simulation/` (10 markets x 3 products, 100 x 3, 100 x 5). Variables: `shares` (J x M), `P_opt` (J x M prices), `x1` (product characteristics, 3 cols), `w` (cost shifter), `Z` (instrument), `xi_all` (demand shock), `eta` (cost shock), `alphas` (M x 500 draws of the price coefficient). Product-level vectors are stacked market by market (assumed), matching `vec(shares)`.

In [ ]:
using MAT, DataFrames, Statistics, StatsBase, Plots, Printf
raw = Dict(
    "10markets3products"  => matread("../build/input/public/simulation/10markets3products.mat"),
    "100markets3products" => matread("../build/input/public/simulation/100markets3products.mat"),
    "100markets5products" => matread("../build/input/public/simulation/100markets5products.mat"),
)
Base.active_project()

In [ ]:
function product_df(d)
    J, M = size(d["shares"])
    df = DataFrame(market = repeat(1:M, inner = J), product = repeat(1:J, outer = M),
                   share = vec(d["shares"]), price = vec(d["P_opt"]),
                   x1_1 = d["x1"][:, 1], x1_2 = d["x1"][:, 2], x1_3 = d["x1"][:, 3],
                   w = vec(d["w"]), Z = vec(d["Z"]), xi = vec(d["xi_all"]), eta = vec(d["eta"]))
    df.outside = 1 .- combine(groupby(df, :market), :share => sum => :s).s[df.market]
    df
end
data = Dict(k => product_df(v) for (k, v) in raw)
DataFrame(dataset = collect(keys(data)), obs = [nrow(data[k]) for k in keys(data)],
          markets = [length(unique(data[k].market)) for k in keys(data)],
          products = [length(unique(data[k].product)) for k in keys(data)],
          alpha_draws = [size(raw[k]["alphas"], 2) for k in keys(data)])

## Summary statistics by dataset

In [ ]:
function desc(df)
    vars = [:share, :outside, :price, :x1_1, :x1_2, :x1_3, :w, :Z, :xi, :eta]
    DataFrame(variable = vars,
        mean = [mean(df[!, v]) for v in vars], std = [std(df[!, v]) for v in vars],
        min = [minimum(df[!, v]) for v in vars], p50 = [median(df[!, v]) for v in vars],
        max = [maximum(df[!, v]) for v in vars])
end
for k in sort(collect(keys(data)))
    println("\n== ", k, " (N = ", nrow(data[k]), ") ==")
    show(stdout, desc(data[k]); allrows = true, allcols = true, eltypes = false); println()
end

## Missing / non-finite values and share checks

In [ ]:
DataFrame(dataset = sort(collect(keys(data))),
    nonfinite = [sum(.!isfinite.(Matrix(data[k][:, 3:end]))) for k in sort(collect(keys(data)))],
    min_outside = [minimum(data[k].outside) for k in sort(collect(keys(data)))],
    mean_outside = [mean(data[k].outside) for k in sort(collect(keys(data)))],
    nonpositive_share = [sum(data[k].share .<= 0) for k in sort(collect(keys(data)))])

## Correlations (largest dataset, 100 markets x 5 products)

In [ ]:
df = data["100markets5products"]
vars = [:share, :price, :x1_1, :x1_2, :x1_3, :w, :Z, :xi, :eta]
C = cor(Matrix(df[:, vars]))
DataFrame(round.(C, digits = 2), string.(vars)) |> d -> insertcols(d, 1, :var => vars)

## Consumer price coefficients (`alphas`)

In [ ]:
DataFrame(dataset = sort(collect(keys(raw))),
    mean = [mean(raw[k]["alphas"]) for k in sort(collect(keys(raw)))],
    std = [std(raw[k]["alphas"]) for k in sort(collect(keys(raw)))],
    min = [minimum(raw[k]["alphas"]) for k in sort(collect(keys(raw)))],
    max = [maximum(raw[k]["alphas"]) for k in sort(collect(keys(raw)))])

## Distributions

In [ ]:
default(size = (900, 300))
ks = sort(collect(keys(data)))
plot(histogram(data[ks[2]].price, title = "price", legend = false),
     histogram(data[ks[2]].share, title = "share", legend = false),
     histogram(vec(raw[ks[2]]["alphas"]), title = "alphas", legend = false), layout = (1, 3))